---

## 📚 快速参考

### 常用命令

```bash
# 激活虚拟环境
source venv/bin/activate

# 下载CycleGAN数据集
bash ./datasets/download_cyclegan_dataset.sh horse2zebra

# 下载pix2pix数据集
bash ./datasets/download_pix2pix_dataset.sh facades

# 下载预训练模型
bash ./scripts/download_pix2pix_model.sh facades_label2photo

# 测试模型
python test.py --dataroot ./datasets/horse2zebra --name horse2zebra_pretrained --model cycle_gan

# 查看帮助
python train.py --help
python test.py --help
```

### GPU故障排除

如果遇到显存不足错误:
```bash
python test.py ... --gpu_ids -1  # 使用CPU
python test.py ... --batch_size 1  # 减少批次大小
python test.py ... --netG resnet_6blocks  # 使用更小的网络
```

### 更多资源
- 📖 [详细配置指南](./LOCAL_SETUP.md)
- 📄 [数据集文档](./docs/datasets.md)
- 💡 [训练技巧](./docs/tips.md)
- ❓ [常见问题解答](./docs/qa.md)

---

**⚠️ 提示**: 首次运行时可能需要下载数百MB的数据。请确保有稳定的网络连接。

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

# 获取结果目录
results_dir = f"./results/{config['name']}/test_latest/images"

if os.path.exists(results_dir):
    img_files = sorted([f for f in os.listdir(results_dir) if f.endswith('.png')])
    print(f"✅ 找到 {len(img_files)} 个生成的图像")
    
    # 显示前N个结果
    n_show = min(6, len(img_files))
    fig, axes = plt.subplots(2, n_show, figsize=(15, 6))
    
    # CycleGAN结果的图像类型
    img_types = ['real_A', 'fake_B'] if test_type == 'cyclegan' else ['real_A', 'fake_B', 'real_B']
    
    for idx in range(n_show):
        for img_type in img_types:
            # 查找匹配的图像
            matching_imgs = [f for f in img_files if f'{idx:03d}_{img_type}.png' in f]
            if matching_imgs:
                img_path = os.path.join(results_dir, matching_imgs[0])
                img = plt.imread(img_path)
                
                if test_type == 'cyclegan':
                    row = 0 if 'real_A' in img_type else 1
                else:
                    row = list(['real_A', 'fake_B', 'real_B']).index(img_type)
                
                if test_type == 'cyclegan':
                    axes[row, idx].imshow(img)
                    axes[row, idx].set_title(f'{img_type}')
                    axes[row, idx].axis('off')
    
    # pix2pix显示
    if test_type == 'pix2pix':
        fig, axes = plt.subplots(3, n_show, figsize=(15, 9))
        for idx in range(n_show):
            for img_type in img_types:
                matching_imgs = [f for f in img_files if f'{idx:04d}_{img_type}.png' in f]
                if matching_imgs:
                    img_path = os.path.join(results_dir, matching_imgs[0])
                    img = plt.imread(img_path)
                    row = img_types.index(img_type)
                    axes[row, idx].imshow(img)
                    axes[row, idx].set_title(f'{img_type}')
                    axes[row, idx].axis('off')
    
    plt.tight_layout()
    plt.show()
    print("\n图像类型说明:")
    print("  - real_A  : 原始输入图像")
    print("  - fake_B  : 生成的转换后的图像")
    if test_type == 'pix2pix':
        print("  - real_B  : 真实目标图像 (若存在)")
else:
    print(f"❌ 结果目录不存在: {results_dir}")

---

## 结果可视化

加载和显示生成的图像结果。

In [ ]:
# ⚙️ 测试配置
test_config = {
    # CycleGAN 测试
    'cyclelogan': {
        'dataroot': './datasets/horse2zebra',
        'name': 'horse2zebra_pretrained',
        'model': 'cycle_gan',
        'num_test': 50,
        'gpu_ids': '-1',  # 使用CPU
    },
    # pix2pix 测试
    'pix2pix': {
        'dataroot': './datasets/facades',
        'direction': 'BtoA',
        'model': 'pix2pix',
        'name': 'facades_label2photo_pretrained',
        'num_test': 50,
        'gpu_ids': '-1',  # 使用CPU
    }
}

# 选择要运行的模型测试
test_type = 'cyclelogan'  # 选择: 'cyclegan' 或 'pix2pix'
config = test_config[test_type]

print(f"🎯 运行 {test_type.upper()} 测试")
print("="*60)
print(f"  数据集: {config['dataroot']}")
print(f"  模型: {config['name']}")
print(f"  测试图像数: {config['num_test']}")
print("="*60)

# 构建测试命令
test_args = ' '.join([f'--{k} {v}' for k, v in config.items()])
test_cmd = f"python test.py {test_args}"

print(f"\n💻 运行命令:")
print(f"  {test_cmd}")
print("\n运行测试...")
print("="*60)

# 运行测试 (注意: 这可能需要几分钟)
result = subprocess.run(test_cmd, shell=True, capture_output=False)

if result.returncode == 0:
    print("\n✅ 测试完成!")
    print(f"📂 结果保存到: ./results/{config['name']}/test_latest/")
else:
    print("\n⚠️ 测试过程中可能出现错误")

---

## 运行模型测试

使用预训练模型对数据集进行推理。这不需要GPU且速度很快。

In [ ]:
# ⚙️ 训练参数配置
train_config = {
    # 基本参数
    'model': 'cycle_gan',              # 模型类型: cycle_gan, pix2pix
    'dataroot': './datasets/horse2zebra',  # 数据集路径
    'name': 'horse2zebra_local_test',  # 模型名称，用于保存
    'direction': None,                 # 转换方向 (pix2pix只用): AtoB, BtoA
    
    # 训练参数
    'n_epochs': 5,                     # 训练周期数 (快速测试用5，正式训练用200)
    'n_epochs_decay': 5,               # 学习率衰减周期
    'batch_size': 1,                   # 批次大小 (根据GPU内存调整)
    'lr': 0.0002,                      # 初始学习率
    'beta1': 0.5,                      # Adam优化器参数
    
    # 显示和保存频率
    'display_freq': 100,               # 每N个迭代显示一次
    'print_freq': 100,                 # 每N个迭代打印一次
    'save_latest_freq': 5000,          # 每N个迭代保存最新模型
    'save_epoch_freq': 1,              # 每N个周期保存模型
    
    # GPU 配置
    'gpu_ids': '0',                    # GPU ID列表，-1表示CPU
    
    # 网络配置
    'netG': 'resnet_6blocks',          # 生成器: resnet_6blocks, resnet_9blocks, unet_256
    'netD': 'basic',                   # 判别器: basic, n_layers, pixel
    
    # 显示配置
    'display_id': -1,                  # 是否使用visdom显示 (-1表示不使用)
    'no_flip': False,                  # 不进行数据增强
}

# 转为命令行参数
def dict_to_args(config):
    args = []
    for key, value in config.items():
        if value is None:
            continue
        if isinstance(value, bool):
            if value:
                args.append(f'--{key}')
        else:
            args.append(f'--{key} {value}')
    return ' '.join(args)

train_args = dict_to_args(train_config)

print("📋 训练配置:")
print("="*60)
for key, value in train_config.items():
    if value is not None:
        print(f"  {key:20} : {value}")
print("\n💾 保存位置: ./checkpoints/{}/".format(train_config['name']))
print("="*60)

---

## 配置训练参数

下面配置用于训练模型的参数。根据您的硬件和需要选择合适的参数。

In [ ]:
# ⚙️ 配置下载的预训练模型
CYCLEGAN_MODEL = 'horse2zebra'         # CycleGAN 预训练模型
PIX2PIX_MODEL = 'facades_label2photo'  # pix2pix 预训练模型

# 下载 CycleGAN 预训练模型
print(f"📥 正在下载 CycleGAN 预训练model: {CYCLEGAN_MODEL}")
print("="*60)

cmd = f"bash ./scripts/download_cyclegan_model.sh {CYCLEGAN_MODEL}"
try:
    result = subprocess.run(cmd, shell=True, capture_output=False, text=True)
    if result.returncode == 0:
        print(f"✅ {CYCLEGAN_MODEL} 预训练模型下载完成!")
except Exception as e:
    print(f"⚠️  下载可能出错或模型已存在: {e}")

# 下载 pix2pix 预训练模型
print(f"\n📥 正在下载 pix2pix 预训练模型: {PIX2PIX_MODEL}")
print("="*60)

cmd = f"bash ./scripts/download_pix2pix_model.sh {PIX2PIX_MODEL}"
try:
    result = subprocess.run(cmd, shell=True, capture_output=False, text=True)
    if result.returncode == 0:
        print(f"✅ {PIX2PIX_MODEL} 预训练模型下载完成!")
except Exception as e:
    print(f"⚠️  下载可能出错或模型已存在: {e}")

# 检查下载的模型
print("\n✅ 已下载的模型:")
checkpoints_dir = './checkpoints'
if os.path.exists(checkpoints_dir):
    for model_dir in os.listdir(checkpoints_dir):
        model_path = os.path.join(checkpoints_dir, model_dir)
        if os.path.isdir(model_path):
            net_g = os.path.join(model_path, 'latest_net_G.pt')
            if os.path.exists(net_g):
                print(f"  📦 {model_dir}/ - latest_net_G.pt ({os.path.getsize(net_g) / (1024*1024):.1f} MB)")

---

## 下载预训练模型

预训练模型允许您立即运行推理而无需训练。可用的模型包括:

### CycleGAN 预训练模型
- `horse2zebra_pretrained`
- `apple2orange_pretrained`
- `vangogh_pretrained`
- `monet_photo_pretrained`

### pix2pix 预训练模型
- `facades_label2photo_pretrained`
- `sat2map_pretrained`
- `map2sat_pretrained`
- `edges2shoes_pretrained`

In [ ]:
# 下载 pix2pix 数据集
print(f"\n📥 正在下载 pix2pix 数据集: {PIX2PIX_DATASET}")
print("="*60)

cmd = f"bash ./datasets/download_pix2pix_dataset.sh {PIX2PIX_DATASET}"
try:
    result = subprocess.run(cmd, shell=True, capture_output=False, text=True)
    if result.returncode == 0:
        print(f"✅ {PIX2PIX_DATASET} 数据集下载完成!")
    else:
        print(f"⚠️ 数据集下载可能出错")
except Exception as e:
    print(f"❌ 错误: {e}")

# 检查下载的数据集
print("\n📂 已下载的数据集:")
datasets_dir = './datasets'
if os.path.exists(datasets_dir):
    for item in os.listdir(datasets_dir):
        path = os.path.join(datasets_dir, item)
        if os.path.isdir(path) and item not in ['bibtex', 'download_scripts']:
            size = sum(os.path.getsize(os.path.join(dirpath, filename)) 
                      for dirpath, dirnames, filenames in os.walk(path)
                      for filename in filenames)
            size_mb = size / (1024 * 1024)
            print(f"  ✅ {item}/ (~{size_mb:.1f} MB)")

In [ ]:
import subprocess
import time

# ⚙️ 配置下载学的数据集
# 选项: horse2zebra, apple2orange, facades, cityscapes, maps, 等
CYCLEGAN_DATASET = 'horse2zebra'  # 最快的选项
PIX2PIX_DATASET = 'facades'        # 推荐选项

# 下载 CycleGAN 数据集
print(f"📥 正在下载 CycleGAN 数据集: {CYCLEGAN_DATASET}")
print("="*60)

cmd = f"bash ./datasets/download_cyclegan_dataset.sh {CYCLEGAN_DATASET}"
try:
    result = subprocess.run(cmd, shell=True, capture_output=False, text=True)
    if result.returncode == 0:
        print(f"✅ {CYCLEGAN_DATASET} 数据集下载完成!")
    else:
        print(f"⚠️ 数据集下载可能出错")
except Exception as e:
    print(f"❌ 错误: {e}")

---

## 下载数据集

选择要下载的数据集。支持的数据集包括:

### CycleGAN 数据集
- `horse2zebra` (最小，~1.2GB)
- `apple2orange` 
- `summer2winter_yosemite`
- `monet_photo`
- `maps`
- `cityscapes`

### pix2pix 数据集
- `facades` (较小，推荐，~500MB)
- `cityscapes`
- `maps`
- `edges2handbags`
- `edges2shoes`

In [ ]:
# 验证依赖安装状态
print("检查依赖安装状态...\n")

required_packages = {
    'torch': 'PyTorch',
    'torchvision': 'TorchVision',
    'numpy': 'NumPy',
    'PIL': 'Pillow',
    'skimage': 'scikit-image',
    'dominate': 'dominate',
}

installed = []
missing = []

for package, name in required_packages.items():
    try:
        __import__(package)
        installed.append(name)
        print(f"✅ {name}")
    except ImportError:
        missing.append(name)
        print(f"❌ {name}")

if missing:
    print(f"\n⚠️  缺失库: {', '.join(missing)}")
    print("请运行: pip install -r requirements.txt")
else:
    print(f"\n✅ 所有依赖已安装!")

In [ ]:
# 设置项目目录
project_dir = '/Users/leon/Desktop/cycleGAN'
os.chdir(project_dir)
print(f"当前工作目录: {os.getcwd()}")

# 列出主要源文件
print("\n✅ 项目结构:")
for item in os.listdir('.'):
    if os.path.isdir(item) and not item.startswith('.'):
        print(f"  📁 {item}/")
    elif item.endswith(('.py', '.txt', '.yml', '.md')):
        print(f"  📄 {item}")

## 克隆和安装依赖

如果您还没有克隆仓库，请运行以下代码。**注**: 如果您已经在项目目录中，可以跳过此步骤。

In [ ]:
import os
import sys
import subprocess

# 检查Python版本
print(f"✅ Python 版本: {sys.version.split()[0]}")

# 检查关键库
try:
    import torch
    print(f"✅ PyTorch 版本: {torch.__version__}")
    print(f"   GPU 可用: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"   GPU 名称: {torch.cuda.get_device_name(0)}")
except:
    print("❌ PyTorch 未安装")

try:
    import torchvision
    print(f"✅ TorchVision 版本: {torchvision.__version__}")
except:
    print("❌ TorchVision 未安装")

try:
    import numpy
    print(f"✅ NumPy 版本: {numpy.__version__}")
except:
    print("❌ NumPy 未安装")

try:
    from PIL import Image
    print(f"✅ Pillow 已安装")
except:
    print("❌ Pillow 未安装")

## 📚 目录
1. [环境检查](#环境检查)
2. [克隆和安装依赖](#克隆和安装依赖)
3. [下载数据集](#下载数据集)
4. [下载预训练模型](#下载预训练模型)
5. [配置训练参数](#配置训练参数)
6. [运行模型测试](#运行模型测试)
7. [结果可视化](#结果可视化)

---

## 环境检查

# 🚀 本地 CycleGAN 和 pix2pix 环境配置指南

本notebook将指导您一步步配置本地环境，下载数据集，训练或测试图像转换模型。

## 系统要求
- **Python**: 3.7+
- **GPU** (推荐): CUDA 11.8+ 或 Apple Silicon
- **磁盘空间**: 至少 10GB
- **内存**: 至少 8GB (16GB 推荐)